# Notebook 2 — Train và đánh giá mô hình

Đọc `ket_qua/dataset_mat.csv` do **`01_trich_dac_trung.ipynb`** tạo ra (chạy notebook 1 trước).
Chạy lại notebook này chỉ mất khoảng nửa phút.

## 1. Cấu hình và đọc dữ liệu

In [ ]:
import random
import time
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GOC = Path.cwd() if (Path.cwd() / "du_lieu").exists() else Path.cwd() / "ml_report"
KQ = GOC / "ket_qua"
assert (KQ / "dataset_mat.csv").exists(), "Chưa có ket_qua/dataset_mat.csv — chạy 01_trich_dac_trung.ipynb trước"

SEED = 42
LFW_TI_LE_TRAIN = 0.7
NGUONG_COSINE = 0.5  # accept_threshold của app
NGUONG_THAT = 0.8  # antispoof_threshold của app
LA = "NGUOI_LA"
E = [f"e{i}" for i in range(512)]
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv(KQ / "dataset_mat.csv")
loai = pd.read_csv(KQ / "anh_bi_loai.csv")
gm_path = KQ / "gia_mao.csv"
gia_mao = pd.read_csv(gm_path) if gm_path.exists() and gm_path.stat().st_size > 1 else pd.DataFrame()
so_anh = df.pivot_table(index="nguoi", columns="buoi", values="file", aggfunc="count", fill_value=0)
print(f"{len(df)} ảnh mặt, {len(gia_mao)} ảnh giả mạo")

## 2. Chia tập train / test

- **Train:** buổi 1–2 của P1…P5 + 70% số danh tính LFW.
- **Test:** buổi 3 + 30% danh tính LFW còn lại.

Không dùng `train_test_split` ngẫu nhiên: các khung liền nhau trong một video gần như giống hệt, chia ngẫu nhiên làm kết quả ảo.
Khi tìm tham số (GridSearch) và chọn ngưỡng, kiểm định chéo cũng chia theo buổi (train buổi 1 → thử buổi 2 và ngược lại).

In [ ]:
ten_lfw = sorted(df.loc[df.buoi == "lfw", "clip"].unique())
random.Random(SEED).shuffle(ten_lfw)
lfw_train = set(ten_lfw[: round(LFW_TI_LE_TRAIN * len(ten_lfw))])
# Mỗi người lạ train được gán vào nhóm buoi1 hoặc buoi2 để kiểm định chéo theo buổi
nhom_lfw = {t: ("buoi1", "buoi2")[k % 2] for k, t in enumerate(sorted(lfw_train))}

la_train = df.buoi.isin(["buoi1", "buoi2"]) | ((df.buoi == "lfw") & df["clip"].isin(lfw_train))
la_test = (df.buoi == "buoi3") | ((df.buoi == "lfw") & ~df["clip"].isin(lfw_train))
tap = np.where(la_train, "train", np.where(la_test, "test", ""))

tr, te = df[la_train].reset_index(drop=True), df[la_test].reset_index(drop=True)
Xtr, ytr = tr[E].to_numpy(np.float32), tr.nguoi.to_numpy()
Xte, yte = te[E].to_numpy(np.float32), te.nguoi.to_numpy()
gtr = np.where(tr.buoi == "lfw", tr["clip"].map(nhom_lfw), tr.buoi)

LOP = sorted(set(ytr) - {LA})  # các danh tính đã đăng ký
NHAN = LOP + [LA]
chia = pd.crosstab(df.nguoi, tap).drop(columns="", errors="ignore")
chia.to_csv(KQ / "chia_du_lieu.csv")
display(chia)
khong_test = [n for n in LOP if n not in set(yte)]
if khong_test:
    print("⚠ Chưa có ảnh test (buổi 3) cho:", ", ".join(khong_test))
print(f"Train {len(tr)} ảnh, test {len(te)} ảnh. Danh tính đã đăng ký: {LOP}")

## 3. Độ đo

Giống cách app hoạt động: sinh viên **khai là Pi**, hệ thống so điểm của Pi với ngưỡng τ.
Mỗi ảnh test × mỗi danh tính Pi là một **lần thử**: thử của chính chủ (ảnh đúng là Pi) hoặc của người khác.

- **FAR** = số lần thử của người khác bị chấp nhận / số lần thử của người khác → tỉ lệ điểm danh hộ lọt qua.
- **FRR** = số lần thử của chính chủ bị từ chối / số lần thử của chính chủ.
- **EER**: giá trị tại ngưỡng mà FAR = FRR.

Nhãn dự đoán (để tính Accuracy, F1, ma trận nhầm lẫn): danh tính có điểm cao nhất nếu điểm ≥ τ, ngược lại `NGUOI_LA`.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, roc_curve

TAUS = np.linspace(0, 1, 1001)


def la_chinh_chu(y):
    return np.array(LOP)[None, :] == np.asarray(y)[:, None]


def far_frr(S, y, taus=TAUS):
    g = la_chinh_chu(y)
    chu, khac = S[g], S[~g]
    far = (khac[None, :] >= np.atleast_1d(taus)[:, None]).mean(1)
    frr = (chu[None, :] < np.atleast_1d(taus)[:, None]).mean(1)
    return far, frr


def eer(S, y):
    far, frr = far_frr(S, y)
    k = np.argmin(np.abs(far - frr))
    return (far[k] + frr[k]) / 2, TAUS[k]


def quyet_dinh(S, tau):
    j = S.argmax(1)
    return np.where(S.max(1) >= tau, np.array(LOP)[j], LA)


# Tự kiểm tra nhỏ: 2 danh tính, ảnh P1 điểm cao ở cột P1
_LOP, LOP = LOP, ["P1", "P2"]
_S = np.array([[0.9, 0.1], [0.2, 0.3], [0.6, 0.7]])
assert list(quyet_dinh(_S, 0.5)) == ["P1", LA, "P2"]
_far, _frr = far_frr(_S, ["P1", "P2", LA], 0.5)
assert np.isclose(_far[0], 2 / 4) and np.isclose(_frr[0], 1 / 2)  # khác: .2,.6,.7 & .1 → 2/4; chính chủ: .9,.3 → 1/2
LOP = _LOP

## 4. Train các mô hình

| Mô hình | Điểm cho danh tính Pi | Ngưỡng τ |
|---|---|---|
| Cosine (app) | cosine với vector trung bình của Pi trong tập train | 0,5 (ngưỡng app đang dùng) |
| Cosine (τ từ train) | như trên | Điểm FAR = FRR, chỉ dùng tập train (xem `chon_nguong_cosine`) |
| k-NN | xác suất lớp Pi | Điểm FAR = FRR, chỉ dùng tập train (xem `chon_nguong`) |
| MLP | xác suất lớp Pi | như trên |
| **SVM** (RBF, Platt scaling) | xác suất lớp Pi | như trên |

Ngưỡng τ được chọn sao cho cân bằng giữa từ chối nhầm chính chủ (train buổi này, thử buổi kia) và nhận nhầm **người lạ quay cùng điều kiện** (bỏ hẳn từng người khỏi tập train). Người lạ LFW không dùng để chọn τ vì ảnh LFW khác kiểu ảnh điện thoại, quá dễ từ chối.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

import warnings

# scikit-learn 1.9 báo SVC(probability=True) sắp bỏ (xoá ở 1.11; requirements.txt đã chặn < 1.11)
warnings.filterwarnings("ignore", message="The `probability` parameter")

cv = LeaveOneGroupOut()

t0 = time.perf_counter()
grid = GridSearchCV(
    SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED),
    {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.5, 1, 2]},
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
)
grid.fit(Xtr, ytr, groups=gtr)
print(f"GridSearch xong sau {time.perf_counter() - t0:.1f} s. Tham số tốt nhất: {grid.best_params_}, F1-macro CV = {grid.best_score_:.4f}")
gs = pd.DataFrame(grid.cv_results_)[["param_C", "param_gamma", "mean_test_score", "std_test_score", "rank_test_score"]]
gs.sort_values("rank_test_score").to_csv(KQ / "gridsearch_svm.csv", index=False)
display(gs.sort_values("rank_test_score").head(8))


def diem_lop(proba, classes):
    cot = {c: k for k, c in enumerate(classes)}
    return np.stack([proba[:, cot[c]] if c in cot else np.zeros(len(proba)) for c in LOP], 1)


def chon_nguong(mh):
    """τ tại điểm FAR = FRR, chỉ dùng tập train:
    - chính chủ: train buổi này, thử ảnh của chính người đó ở buổi kia;
    - người lạ CÙNG điều kiện quay: bỏ hẳn một người Pi khỏi tập train, lấy điểm cao nhất mà ảnh Pi nhận được.
    Không dùng người lạ LFW để chọn τ: ảnh LFW khác kiểu ảnh điện thoại nên quá dễ từ chối, làm τ thấp
    (thử nghiệm trước: τ = 0,16 và SVM nhận nhầm 26% người lạ cùng điều kiện)."""
    chu, la = [], []
    for a, b in cv.split(Xtr, ytr, gtr):
        m = clone(mh).fit(Xtr[a], ytr[a])
        b = b[np.isin(ytr[b], [c for c in m.classes_ if c != LA])]  # bỏ người vắng mặt trong lượt train
        s = diem_lop(m.predict_proba(Xtr[b]), m.classes_)
        chu.append(s[np.arange(len(b)), [LOP.index(v) for v in ytr[b]]])
    for h in LOP:
        m = clone(mh).fit(Xtr[ytr != h], ytr[ytr != h])
        la.append(diem_lop(m.predict_proba(Xtr[ytr == h]), m.classes_).max(1))  # cột Pi bằng 0 vì m không biết Pi
    chu, la = np.concatenate(chu), np.concatenate(la)
    far = (la[None, :] >= TAUS[:, None]).mean(1)
    frr = (chu[None, :] < TAUS[:, None]).mean(1)
    return TAUS[np.argmin(np.abs(far - frr))]


MO_HINH = {
    "k-NN": KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance"),
    "MLP": MLPClassifier(hidden_layer_sizes=(256,), max_iter=500, random_state=SEED),
    "SVM": grid.best_estimator_,
}

def tam(X, y, lop):
    """Vector trung bình (chuẩn hoá L2) của từng danh tính — 'mẫu đăng ký' như app."""
    C = np.stack([X[y == n].mean(0) for n in lop])
    return C / np.linalg.norm(C, axis=1, keepdims=True)


def chon_nguong_cosine():
    """Như chon_nguong nhưng cho cosine: chính chủ = mẫu từ buổi này, thử buổi kia;
    người lạ cùng điều kiện = bỏ hẳn từng người khỏi mẫu, lấy cosine cao nhất ảnh của họ nhận được."""
    chu, la = [], []
    for g, k in (("buoi1", "buoi2"), ("buoi2", "buoi1")):
        a = (gtr == g) & (ytr != LA)
        lop = [n for n in LOP if (ytr[a] == n).any()]
        b = np.flatnonzero((gtr == k) & np.isin(ytr, lop))
        s = Xtr[b] @ tam(Xtr[a], ytr[a], lop).T
        chu.append(s[np.arange(len(b)), [lop.index(v) for v in ytr[b]]])
    for h in LOP:
        a = (ytr != h) & (ytr != LA)
        la.append((Xtr[ytr == h] @ tam(Xtr[a], ytr[a], [n for n in LOP if n != h]).T).max(1))
    chu, la = np.concatenate(chu), np.concatenate(la)
    far = (la[None, :] >= TAUS[:, None]).mean(1)
    frr = (chu[None, :] < TAUS[:, None]).mean(1)
    return TAUS[np.argmin(np.abs(far - frr))]


ket_qua = {}

# Cosine: một mẫu trung bình cho mỗi người, như app; hai ngưỡng
t0 = time.perf_counter()
C = tam(Xtr, ytr, LOP)
t_train = time.perf_counter() - t0
t0 = time.perf_counter()
S = Xte @ C.T
t_du_doan = time.perf_counter() - t0
ket_qua["Cosine (app)"] = dict(S=S, tau=NGUONG_COSINE, t_train=t_train, t_du_doan=t_du_doan)
ket_qua["Cosine (τ từ train)"] = dict(S=S, tau=chon_nguong_cosine(), t_train=t_train, t_du_doan=t_du_doan)
print(f"Ngưỡng cosine chọn từ tập train: {ket_qua['Cosine (τ từ train)']['tau']:.3f} (app đang dùng {NGUONG_COSINE})")

for ten, mh in MO_HINH.items():
    tau = chon_nguong(mh)
    t0 = time.perf_counter()
    mh = clone(mh).fit(Xtr, ytr)
    t_train = time.perf_counter() - t0
    t0 = time.perf_counter()
    S = diem_lop(mh.predict_proba(Xte), mh.classes_)
    ket_qua[ten] = dict(S=S, tau=tau, t_train=t_train, t_du_doan=time.perf_counter() - t0, mo_hinh=mh)
print("Đã train xong các mô hình.")

## 5. So sánh trên tập test

Accuracy, F1, FAR, FRR phụ thuộc ngưỡng τ. **EER và AUC không phụ thuộc ngưỡng**, nên hai dòng cosine (cùng điểm số, khác ngưỡng) có EER và AUC giống hệt nhau, và chung một đường ROC.

In [ ]:
bang = []
for ten, r in ket_qua.items():
    r["pred"] = quyet_dinh(r["S"], r["tau"])
    p, rc, f1, _ = precision_recall_fscore_support(yte, r["pred"], labels=sorted(set(yte)), average="macro", zero_division=0)
    far, frr = far_frr(r["S"], yte, r["tau"])
    e, _ = eer(r["S"], yte)
    bang.append({
        "Mô hình": ten,
        "Accuracy": accuracy_score(yte, r["pred"]),
        "Precision (macro)": p,
        "Recall (macro)": rc,
        "F1 (macro)": f1,
        "FAR": far[0],
        "FRR": frr[0],
        "EER": e,
        "AUC": roc_auc_score(la_chinh_chu(yte).ravel(), r["S"].ravel()),
        "Ngưỡng τ": r["tau"],
        "Train (s)": r["t_train"],
        "Dự đoán (ms/ảnh)": 1000 * r["t_du_doan"] / len(yte),
    })
so_sanh = pd.DataFrame(bang).set_index("Mô hình")
so_sanh.round(4).to_csv(KQ / "bang_so_sanh.csv")
display(so_sanh.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for ten, r in ket_qua.items():
    if ten == "Cosine (τ từ train)":
        continue  # cùng điểm số với Cosine (app), chỉ khác ngưỡng → cùng một đường ROC
    fpr, tpr, _ = roc_curve(la_chinh_chu(yte).ravel(), r["S"].ravel())
    nhan = "Cosine (cả 2 ngưỡng)" if ten == "Cosine (app)" else ten
    ax.plot(fpr, tpr, label=f"{nhan} (AUC = {so_sanh.loc[ten, 'AUC']:.4f})")
ax.plot([0, 1], [0, 1], "k:", lw=0.8)
ax.set_xscale("log")
ax.set_xlim(1e-4, 1)
ax.set_xlabel("FAR (FPR, thang log)")
ax.set_ylabel("1 − FRR (TPR)")
ax.set_title("Đường ROC trên tập test")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig(KQ / "roc.png")
plt.show()

## 6. Phân tích chi tiết SVM

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

svm = ket_qua["SVM"]
bc = pd.DataFrame(classification_report(yte, svm["pred"], labels=NHAN, output_dict=True, zero_division=0)).T
bc.round(4).to_csv(KQ / "bao_cao_phan_loai_svm.csv")
display(bc.round(4))

fig, ax = plt.subplots(figsize=(6, 5.5))
ConfusionMatrixDisplay.from_predictions(yte, svm["pred"], labels=NHAN, ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Ma trận nhầm lẫn của SVM trên tập test")
fig.tight_layout()
fig.savefig(KQ / "ma_tran_nham_lan_svm.png")
plt.show()

In [ ]:
far, frr = far_frr(svm["S"], yte)
e, tau_e = eer(svm["S"], yte)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(TAUS, far, label="FAR")
ax.plot(TAUS, frr, label="FRR")
ax.axvline(svm["tau"], color="k", ls="--", label=f"τ đã chọn = {svm['tau']:.3f}")
ax.scatter([tau_e], [e], color="red", zorder=3, label=f"EER (test) = {e:.4f} tại τ = {tau_e:.3f}")
ax.set_xlabel("Ngưỡng τ (xác suất của danh tính được khai)")
ax.set_ylabel("Tỉ lệ")
ax.set_title("FAR và FRR của SVM theo ngưỡng")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(KQ / "far_frr.png")
plt.show()

In [ ]:
sai = np.flatnonzero(svm["pred"] != yte)
print(f"SVM đoán sai {len(sai)}/{len(yte)} ảnh test")
if len(sai):
    chon = sai[:: max(1, len(sai) // 12)][:12]
    fig, ax = plt.subplots(2, 6, figsize=(12, 4.8))
    for a in ax.ravel():
        a.axis("off")
    for a, k in zip(ax.ravel(), chon):
        a.imshow(cv2.imread(str(GOC / te.file[k]))[..., ::-1])
        a.set_title(f"Thật: {yte[k]}\nĐoán: {svm['pred'][k]} ({svm['S'][k].max():.2f})", fontsize=8)
    fig.suptitle("Ảnh test SVM đoán sai")
    fig.tight_layout()
    fig.savefig(KQ / "anh_doan_sai.png")
    plt.show()

### Dự đoán thử một ảnh test

In [ ]:
k = np.random.default_rng().integers(len(te))
p = svm["S"][k]
plt.figure(figsize=(2.5, 2.8))
plt.imshow(cv2.imread(str(GOC / te.file[k]))[..., ::-1])
plt.axis("off")
plt.title(f"Thật: {yte[k]} — Dự đoán: {svm['pred'][k]}", fontsize=9)
plt.show()
print(te.file[k])
print("Xác suất từng danh tính:", {n: round(float(v), 4) for n, v in zip(LOP, p)}, f"— ngưỡng τ = {svm['tau']:.3f}")

### Kiểm tra thêm: người lạ cùng điều kiện quay

Người lạ trong tập test đều là ảnh LFW (ảnh báo chí), khác hẳn ảnh camera điện thoại. Mô hình có thể từ chối họ nhờ **kiểu ảnh** chứ không nhờ **khuôn mặt**, nên FAR ở bảng so sánh có thể quá lạc quan.

Kiểm tra: lần lượt coi từng người P1…P5 là người lạ — train lại khi **không có** người đó (giữ nguyên tham số và ngưỡng τ), rồi đưa ảnh buổi 3 của họ vào. Tỉ lệ bị nhận nhầm thành một sinh viên khác là **FAR với người lạ cùng điều kiện quay**, sát với tình huống điểm danh hộ thật hơn.

In [ ]:
dong = []
for h in LOP:
    x = Xte[yte == h]
    if not len(x):
        continue
    giu = ytr != h
    lop_h = [n for n in LOP if n != h]
    cos_h = (x @ tam(Xtr[giu & (ytr != LA)], ytr[giu & (ytr != LA)], lop_h).T).max(1)
    hang = {"Người bị coi là người lạ": h, "Số ảnh": len(x)}
    for ten, r in ket_qua.items():
        if "mo_hinh" not in r:  # cosine
            hang[ten] = float((cos_h >= r["tau"]).mean())
        else:
            m = clone(r["mo_hinh"]).fit(Xtr[giu], ytr[giu])
            S_h = diem_lop(m.predict_proba(x), m.classes_)[:, [LOP.index(n) for n in lop_h]]
            hang[ten] = float((S_h.max(1) >= r["tau"]).mean())
    dong.append(hang)
nguoi_la = pd.DataFrame(dong).set_index("Người bị coi là người lạ").astype(float)
nguoi_la.loc["Trung bình"] = nguoi_la.mean()
nguoi_la.round(4).to_csv(KQ / "nguoi_la_cung_dieu_kien.csv")
display(nguoi_la.round(4))

## 7. Chống giả mạo (MiniFASNetV2 + MiniFASNetV1SE, như app)

Phần này đánh giá **riêng mô hình, theo từng khung**: ảnh giả mạo không bị lọc chất lượng (ảnh người thật đã qua lọc ở notebook 1). Số liệu cho hệ thống thật nằm ở mục "theo lượt điểm danh" ngay sau.

Theo ISO/IEC 30107-3, với ngưỡng "mặt thật" của app = 0,8:
- **APCER** (từng kiểu tấn công) = ảnh giả bị nhận là thật / số ảnh giả kiểu đó.
- **BPCER** = ảnh người thật bị nhận là giả / số ảnh người thật.
- **ACER** = (APCER lớn nhất + BPCER) / 2.

In [ ]:
that = df.loc[df.buoi.str.startswith("buoi"), "diem_that"]
if len(gia_mao):
    bpcer = float((that < NGUONG_THAT).mean())
    dong = [{"Loại": "Người thật (bona fide)", "Số ảnh": len(that), "Tỉ lệ lỗi": bpcer, "Độ đo": "BPCER"}]
    for kieu, g in gia_mao.groupby("clip"):
        dong.append({"Loại": kieu, "Số ảnh": len(g), "Tỉ lệ lỗi": float((g.diem_that >= NGUONG_THAT).mean()), "Độ đo": "APCER"})
    gm = pd.DataFrame(dong)
    acer = (gm.loc[gm["Độ đo"] == "APCER", "Tỉ lệ lỗi"].max() + bpcer) / 2
    gm = pd.concat([gm, pd.DataFrame([{"Loại": "Tổng hợp", "Số ảnh": len(that) + len(gia_mao), "Tỉ lệ lỗi": acer, "Độ đo": "ACER"}])])
    gm.round(4).to_csv(KQ / "chong_gia_mao.csv", index=False)
    display(gm.round(4))

    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    bins = np.linspace(0, 1, 41)
    ax.hist(that, bins=bins, alpha=0.6, density=True, label="Người thật")
    for kieu, g in gia_mao.groupby("clip"):
        ax.hist(g.diem_that, bins=bins, alpha=0.6, density=True, label=f"Giả mạo: {kieu}")
    ax.axvline(NGUONG_THAT, color="k", ls="--", label=f"Ngưỡng app = {NGUONG_THAT}")
    ax.set_xlabel('Điểm "mặt thật"')
    ax.set_ylabel("Mật độ")
    ax.legend(fontsize=8)
    ax.set_title("Phân bố điểm chống giả mạo")
    fig.tight_layout()
    fig.savefig(KQ / "chong_gia_mao.png")
    plt.show()
else:
    acer = None
    print("Không có video giả mạo — bỏ qua phần này.")

### Đánh giá theo lượt điểm danh (giống app)

Các con số ở trên tính **theo từng khung**. App không quyết định như vậy (`backend/apps/attendance/face/verify.py`): mỗi lượt điểm danh chụp **6 khung** (2 nhìn thẳng + 4 làm động tác), loại khung kém chất lượng, lấy **3 khung nét nhất** và dùng **điểm "mặt thật" thấp nhất** trong 3 khung đó.

Mô phỏng: mỗi cửa sổ 6 khung liên tiếp (lấy 2 khung/giây) trong một clip là một lượt. Khung giả mạo cũng bị lọc chất lượng như app. Hai điểm đơn giản hoá:
- Cửa sổ chỉ gồm các khung **đã qua lọc chất lượng**, nên lượt nào cũng đủ ≥ 3 khung đạt. Lượt bị app từ chối vì ảnh kém (tối, mờ) không được tính ở đây.
- Bỏ qua bước kiểm tra động tác (coi như kẻ giả mạo vượt qua được), nên APCER ở đây là **cận trên**.

In [ ]:
KHUNG_MOI_LUOT, KHUNG_CHONG_GIA = 6, 3


def diem_luot(bang):
    """Điểm 'mặt thật' của từng lượt: min của 3 khung nét nhất trong mỗi cửa sổ 6 khung liên tiếp."""
    diem = []
    for _, c in bang.sort_values("file").groupby(["nguoi", "buoi", "clip"]):
        for i in range(len(c) - KHUNG_MOI_LUOT + 1):
            diem.append(c.iloc[i : i + KHUNG_MOI_LUOT].nlargest(KHUNG_CHONG_GIA, "do_net").diem_that.min())
    return np.array(diem)


# Tự kiểm tra: 6 khung, 3 khung nét nhất có điểm 0.9, 0.7, 0.95 → điểm lượt 0.7
_t = pd.DataFrame({"nguoi": "X", "buoi": "b", "clip": "c", "file": list("abcdef"),
                   "do_net": [10, 90, 80, 5, 70, 1], "diem_that": [0.1, 0.9, 0.7, 0.2, 0.95, 0.0]})
assert np.allclose(diem_luot(_t), [0.7])

quet = None
if len(gia_mao):
    gm_dat = gia_mao[(gia_mao.rong_mat >= 100) & (gia_mao.do_sang >= 60) & (gia_mao.do_net >= 50)]
    that_df = df[df.buoi.str.startswith("buoi")]
    luot_that = diem_luot(that_df)
    luot_gia = {k: diem_luot(v) for k, v in gm_dat.groupby("clip")}
    print(f"Số lượt: người thật {len(luot_that)}; " + ", ".join(f"{k} {len(v)}" for k, v in luot_gia.items()))

    def ti_le(t):
        bp = float((luot_that < t).mean())
        ap = {f"APCER {k}": float((v >= t).mean()) for k, v in luot_gia.items()}
        return {"BPCER": bp, **ap, "ACER": (max(ap.values()) + bp) / 2}

    quet = pd.DataFrame({t: ti_le(t) for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]}).T
    quet.index.name = "Ngưỡng"
    quet.round(4).to_csv(KQ / "chong_gia_mao_theo_luot.csv")
    display(quet.round(4))

    luoi = np.linspace(0, 1, 101)
    duong = pd.DataFrame([ti_le(t) for t in luoi], index=luoi)
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for cot in duong.columns.drop("ACER"):
        ax.plot(luoi, duong[cot], label=cot)
    ax.axvline(NGUONG_THAT, color="k", ls="--", label=f"Ngưỡng app = {NGUONG_THAT}")
    ax.set_xlabel('Ngưỡng điểm "mặt thật"')
    ax.set_ylabel("Tỉ lệ lỗi (theo lượt)")
    ax.set_title("Chống giả mạo theo lượt điểm danh")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(KQ / "chong_gia_mao_theo_luot.png")
    plt.show()

    # Lỗi nằm ở đâu: tại ngưỡng app, theo từng người
    chi_tiet = [
        {"Người": n, "Nguồn": b, "Số lượt": len(s := diem_luot(v)),
         "Tỉ lệ lỗi": float((s < NGUONG_THAT).mean()) if len(s) else np.nan, "Độ đo": "BPCER"}
        for (n, b), v in that_df.groupby(["nguoi", "buoi"])
    ] + [
        {"Người": n, "Nguồn": k, "Số lượt": len(s := diem_luot(v)),
         "Tỉ lệ lỗi": float((s >= NGUONG_THAT).mean()) if len(s) else np.nan, "Độ đo": "APCER"}
        for (n, k), v in gm_dat.groupby(["nguoi", "clip"])
    ]
    display(pd.DataFrame(chi_tiet).round(3))

## 8. Lưu mô hình và tóm tắt số liệu cho báo cáo

In [ ]:
joblib.dump({"mo_hinh": svm["mo_hinh"], "danh_tinh": LOP, "nguong": svm["tau"]}, KQ / "svm.pkl")

tom_tat = [
    "SỐ LIỆU CHO BÁO CÁO",
    f"Tổng ảnh mặt: {len(df)} (5 người: {int((df.nguoi != LA).sum())}, LFW: {int((df.nguoi == LA).sum())} ảnh / {len(ten_lfw)} người)",
    f"Khung bị loại: {len(loai)} ({loai.ly_do.value_counts().to_dict() if len(loai) else {}})",
    f"Train: {len(tr)} ảnh; Test: {len(te)} ảnh",
    "",
    "Số ảnh mỗi người / buổi:",
    so_anh.to_string(),
    "",
    f"SVM GridSearch: {grid.best_params_}, F1-macro CV = {grid.best_score_:.4f}",
    "",
    "So sánh các mô hình trên tập test:",
    so_sanh.round(4).to_string(),
    "",
    "FAR với người lạ cùng điều kiện quay (bỏ từng người khỏi tập train):",
    nguoi_la.round(4).to_string(),
]
if acer is not None:
    tom_tat += ["", "Chống giả mạo theo từng khung (ngưỡng 0,8):", gm.round(4).to_string(index=False)]
if quet is not None:
    tom_tat += ["", "Chống giả mạo theo lượt điểm danh (như app):", quet.round(4).to_string()]
(KQ / "so_lieu_bao_cao.txt").write_text("\n".join(tom_tat), encoding="utf-8")
print("\n".join(tom_tat))